In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
)

# 1) Load preprocessed data (03-preprocessing output)
data_path = "/files/financial-kpis-analysis-and-distress-prediction/data/processed/compustat_kpis_clean_scaled.csv"
df = pd.read_csv(data_path)

print("Initial shape:", df.shape)
display(df.head())

# 2) Create a better 'distress' variable based on 5 core KPIs
# We use: roa, free_cash_flow, interest_coverage,
#         working_capital_to_assets, retained_earnings_to_assets

required_cols = [
    "roa",
    "free_cash_flow",
    "interest_coverage",
    "working_capital_to_assets",
    "retained_earnings_to_assets",
]

# Keep only rows where all components are available
df_z = df.dropna(subset=required_cols).copy()

# Higher distress_score = worse financial health
df_z["distress_score"] = (
    - df_z["roa"]
    - df_z["free_cash_flow"]
    - df_z["interest_coverage"]
    - df_z["working_capital_to_assets"]
    - df_z["retained_earnings_to_assets"]
)

# Define distressed firms as the 10% worst observations
threshold = df_z["distress_score"].quantile(0.90)
df_z["distress"] = (df_z["distress_score"] >= threshold).astype(int)

print("\nDistribution of distress variable (0 = healthy, 1 = distress):")
print(df_z["distress"].value_counts(normalize=True))


Initial shape: (240308, 21)


,gvkey,datadate,fyear,conm,tic,roa,ebitda_margin,debt_ratio,total_debt_to_equity,current_ratio,...,interest_coverage,cfo_margin,free_cash_flow,asset_turnover,sales_growth,asset_growth,book_to_market,price_to_book,working_capital_to_assets,retained_earnings_to_assets
0,1004,1995-05-31,1994,AAR CORP,AIR,0.226063,0.171086,-0.181664,-0.061026,0.497012,...,0.070501,0.169756,-0.205238,0.283988,-0.192809,-0.215570,-0.003440,-0.148203,0.372367,0.214576
1,1004,1996-05-31,1995,AAR CORP,AIR,0.238559,0.171952,-0.183780,-0.071672,0.456792,...,0.076704,0.172377,-0.188842,0.391643,-0.089662,-0.201567,-0.003440,-0.087432,0.376548,0.215062
2,1004,1997-05-31,1996,AAR CORP,AIR,0.246015,0.173069,-0.203332,-0.122454,0.423344,...,0.083720,0.166723,-0.245287,0.344856,-0.031475,-0.051090,-0.003440,-0.042469,0.377995,0.214125
3,1004,1998-05-31,1997,AAR CORP,AIR,0.255895,0.173950,-0.174871,-0.069742,0.134870,...,0.087443,0.168962,-0.206508,0.406674,0.161531,-0.004039,-0.003440,-0.000877,0.309188,0.213540
4,1004,1999-05-31,1998,AAR CORP,AIR,0.260078,0.174173,-0.174890,-0.081766,0.073524,...,0.085177,0.169287,-0.225733,0.518676,-0.023321,-0.155609,0.138547,-0.096157,0.300008,0.214827



Distribution of distress variable (0 = healthy, 1 = distress):
distress
0    0.899999
1    0.100001
Name: proportion, dtype: float64
